In [1]:
import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path("../data/raw/emowoz")

with open(DATA_DIR / "emowoz-multiwoz.json", encoding="utf-8") as f:
    multiwoz = json.load(f)

with open(DATA_DIR / "emowoz-dialmage.json", encoding="utf-8") as f:
    dialmage = json.load(f)

with open(DATA_DIR / "data-split.json", encoding="utf-8") as f:
    data_split = json.load(f)

In [2]:
print(type(multiwoz))
print(type(dialmage))
print(type(data_split))

print("\nMultiWOZ dialogues:", len(multiwoz))
print("DialMAGE dialogues:", len(dialmage))
print("Split keys:", data_split.keys())

<class 'dict'>
<class 'dict'>
<class 'dict'>

MultiWOZ dialogues: 10438
DialMAGE dialogues: 995
Split keys: dict_keys(['train', 'dev', 'test'])


In [3]:
first_id = next(iter(multiwoz))

print("Dialogue ID:", first_id)
print("Dialogue fields:", multiwoz[first_id].keys())

multiwoz[first_id]["log"][:4]

Dialogue ID: PMUL2335.json
Dialogue fields: dict_keys(['log'])


[{'text': 'How are you doing? Are there any European restaurants in the city center?',
  'emotion': [{'annotator': '68bd033a', 'annotation': 0},
   {'annotator': '13de8dba', 'annotation': 0},
   {'annotator': 'c522c02d', 'annotation': 0},
   {'emotion': 0, 'sentiment': 0}],
  'dialog_act': {'Restaurant-Inform': [['Food', 'european']]},
  'span_info': [['Restaurant-Inform', 'Food', 'european', 8, 8]]},
 {'text': 'Yes, there are 8. What is your price range?',
  'emotion': [],
  'dialog_act': {'Restaurant-Request': [['Price', '?']],
   'Restaurant-Inform': [['Choice', '8']]},
  'span_info': [['Restaurant-Inform', 'Choice', '8', 4, 4]]},
 {'text': 'Can I get the name and address of one of the most inexpensive ones?',
  'emotion': [{'annotator': '68bd033a', 'annotation': 0},
   {'annotator': '13de8dba', 'annotation': 0},
   {'annotator': 'c522c02d', 'annotation': 0},
   {'emotion': 0, 'sentiment': 0}],
  'dialog_act': {'Restaurant-Inform': [['Price', 'cheap']]},
  'span_info': []},
 {'text'

In [4]:
for split_name, split_data in data_split.items():
    print(f"\n{split_name.upper()}")

    for source, ids in split_data.items():
        print(f"{source:10} {len(ids):,}")


TRAIN
multiwoz   8,438
dialmage   796

DEV
multiwoz   1,000
dialmage   100

TEST
multiwoz   1,000
dialmage   100


In [5]:
for split_name, split_data in data_split.items():
    for source, ids in split_data.items():

        counts = Counter(ids)

        duplicates = {
            dialogue_id: count
            for dialogue_id, count in counts.items()
            if count > 1
        }

        print(
            f"{split_name:5} | "
            f"{source:8} | "
            f"{duplicates}"
        )

train | multiwoz | {}
train | dialmage | {'DMAGE3401.json': 2}
dev   | multiwoz | {}
dev   | dialmage | {}
test  | multiwoz | {}
test  | dialmage | {}


In [6]:
EMOTION_LABELS = {
    0: "neutral",
    1: "fearful",
    2: "dissatisfied",
    3: "apologetic",
    4: "abusive",
    5: "excited",
    6: "satisfied",
}

In [7]:
split_dialogues = {}

for split_name, split_data in data_split.items():

    multiwoz_ids = list(
        dict.fromkeys(split_data["multiwoz"])
    )

    dialmage_ids = list(
        dict.fromkeys(split_data["dialmage"])
    )

    split_dialogues[split_name] = {
        "multiwoz": multiwoz_ids,
        "dialmage": dialmage_ids,
    }

    print(
        f"{split_name:5} | "
        f"MultiWOZ: {len(multiwoz_ids):,} | "
        f"DialMAGE: {len(dialmage_ids):,} | "
        f"Total: {len(multiwoz_ids) + len(dialmage_ids):,}"
    )

train | MultiWOZ: 8,438 | DialMAGE: 795 | Total: 9,233
dev   | MultiWOZ: 1,000 | DialMAGE: 100 | Total: 1,100
test  | MultiWOZ: 1,000 | DialMAGE: 100 | Total: 1,100


In [8]:
import pandas as pd

rows = []

sources = {
    "multiwoz": multiwoz,
    "dialmage": dialmage,
}

for split_name, split_data in split_dialogues.items():

    for source_name, dialogue_ids in split_data.items():

        source = sources[source_name]

        for dialogue_id in dialogue_ids:

            dialogue = source[dialogue_id]

            for turn_id, turn in enumerate(dialogue["log"]):

                is_user = turn_id % 2 == 0

                if is_user:
                    emotion_id = turn["emotion"][3]["emotion"]
                    emotion = EMOTION_LABELS[emotion_id]
                else:
                    emotion_id = -1
                    emotion = None

                rows.append({
                    "split": split_name,
                    "source": source_name,
                    "dialogue_id": dialogue_id,
                    "turn_id": turn_id,
                    "speaker": "user" if is_user else "system",
                    "text": turn["text"],
                    "emotion_id": emotion_id,
                    "emotion": emotion,
                })

emowoz_df = pd.DataFrame(rows)

emowoz_df.head(10)

,split,source,dialogue_id,turn_id,speaker,text,emotion_id,emotion
0,train,multiwoz,SNG01856.json,0,user,am looking for a place to to stay that has che...,0,neutral
1,train,multiwoz,SNG01856.json,1,system,"Okay, do you have a specific area you want to ...",-1,NaN
2,train,multiwoz,SNG01856.json,2,user,"no, i just need to make sure it's cheap. oh, a...",0,neutral
3,train,multiwoz,SNG01856.json,3,system,I found 1 cheap hotel for you that includes pa...,-1,NaN
4,train,multiwoz,SNG01856.json,4,user,"Yes, please. 6 people 3 nights starting on tue...",0,neutral
5,train,multiwoz,SNG01856.json,5,system,I am sorry but I wasn't able to book that for ...,-1,NaN
6,train,multiwoz,SNG01856.json,6,user,how about only 2 nights.,0,neutral
7,train,multiwoz,SNG01856.json,7,system,Booking was successful.\nReference number is :...,-1,NaN
8,train,multiwoz,SNG01856.json,8,user,"No, that will be all. Good bye.",0,neutral
9,train,multiwoz,SNG01856.json,9,system,Thank you for using our services.,-1,NaN


In [9]:
for split_name in ["train", "dev", "test"]:

    split_df = emowoz_df[
        emowoz_df["split"] == split_name
    ]

    n_dialogues = split_df["dialogue_id"].nunique()
    n_turns = len(split_df)
    n_user_turns = (split_df["speaker"] == "user").sum()
    n_system_turns = (split_df["speaker"] == "system").sum()

    print(
        f"{split_name:5} | "
        f"{n_dialogues:,} dialogues | "
        f"{n_turns:,} turns | "
        f"{n_user_turns:,} user | "
        f"{n_system_turns:,} system"
    )

train | 9,233 dialogues | 132,948 turns | 66,474 user | 66,474 system
dev   | 1,100 dialogues | 17,018 turns | 8,509 user | 8,509 system
test  | 1,100 dialogues | 17,268 turns | 8,634 user | 8,634 system


In [10]:
alternation_errors = []

for key, dialogue in emowoz_df.groupby(
    ["split", "source", "dialogue_id"]
):

    dialogue = dialogue.sort_values("turn_id")

    for row in dialogue.itertuples():

        expected = (
            "user"
            if row.turn_id % 2 == 0
            else "system"
        )

        if row.speaker != expected:
            alternation_errors.append(
                (key, row.turn_id)
            )

print("Alternation errors:", len(alternation_errors))

Alternation errors: 0


In [11]:
annotation_errors = []

for source_name, source in sources.items():

    for dialogue_id, dialogue in source.items():

        for turn_id, turn in enumerate(dialogue["log"]):

            if turn_id % 2 == 0:
                if len(turn["emotion"]) < 4:
                    annotation_errors.append(
                        (source_name, dialogue_id, turn_id, "user")
                    )

            else:
                if len(turn["emotion"]) != 0:
                    annotation_errors.append(
                        (source_name, dialogue_id, turn_id, "system")
                    )

print("Annotation-pattern errors:", len(annotation_errors))

Annotation-pattern errors: 0


In [12]:
print("Missing text:", emowoz_df["text"].isna().sum())
print(
    "Empty text:",
    emowoz_df["text"].fillna("").str.strip().eq("").sum()
)

Missing text: 0
Empty text: 849


In [13]:
empty_df = emowoz_df[
    emowoz_df["text"].str.strip().eq("")
]

print(f"Empty turns: {len(empty_df):,}")
print(f"Percentage: {len(empty_df) / len(emowoz_df) * 100:.2f}%")

print("\nBy speaker:")
print(empty_df["speaker"].value_counts())

print("\nBy source:")
print(empty_df["source"].value_counts())

print("\nBy split:")
print(empty_df["split"].value_counts())

Empty turns: 849
Percentage: 0.51%

By speaker:
speaker
system    848
user        1
Name: count, dtype: int64

By source:
source
dialmage    849
Name: count, dtype: int64

By split:
split
train    650
test     101
dev       98
Name: count, dtype: int64


In [14]:
for _, empty_row in empty_df.head(5).iterrows():

    context = emowoz_df[
        (emowoz_df["split"] == empty_row["split"]) &
        (emowoz_df["source"] == empty_row["source"]) &
        (emowoz_df["dialogue_id"] == empty_row["dialogue_id"])
    ].sort_values("turn_id")

    turn_id = empty_row["turn_id"]

    context = context[
        context["turn_id"].between(turn_id - 2, turn_id + 2)
    ]

    print(
        f'\n{empty_row["source"]} | '
        f'{empty_row["dialogue_id"]} | '
        f'empty turn {turn_id}'
    )

    for row in context.itertuples():
        print(
            f'{row.turn_id:2} | '
            f'{row.speaker:6} | '
            f'{str(row.emotion):12} | '
            f'{repr(row.text)}'
        )

    print("-" * 80)


dialmage | DMAGE23.json | empty turn 7
 5 | system | nan          | 'What day will you be staying ?'
 6 | user   | neutral      | 'I am looking for a train. The train should depart from leicester. The train should leave on thursday.'
 7 | system | nan          | ''
 8 | user   | neutral      | 'The train should arrive by 20:30. The train should go to stevenage.'
 9 | system | nan          | 'Where did you want to depart from ?'
--------------------------------------------------------------------------------

dialmage | DMAGE32.json | empty turn 1
 0 | user   | neutral      | "Hi, i'm looking for tourist attraction place, located at the center of the city"
 1 | system | nan          | ''
 2 | user   | dissatisfied | "Hi, i'm looking for tourist attraction place, located at the center of the city"
 3 | system | nan          | 'You are welcome . Is there anything else I can help you with today ?'
--------------------------------------------------------------------------------

dialmage |

In [15]:
example = empty_df.iloc[0]

source_data = sources[example["source"]]
raw_turn = source_data[
    example["dialogue_id"]
]["log"][example["turn_id"]]

raw_turn

{'text': '', 'emotion': [], 'dialog_act': {}, 'span_info': []}

In [16]:
empty_dialogues = empty_df[
    ["split", "source", "dialogue_id"]
].drop_duplicates()

print(
    f"Dialogues containing empty turns: "
    f"{len(empty_dialogues):,}"
)

empty_counts_per_dialogue = (
    empty_df
    .groupby(["split", "source", "dialogue_id"])
    .size()
)

print("\nEmpty turns per affected dialogue:")
print(empty_counts_per_dialogue.value_counts().sort_index())

Dialogues containing empty turns: 526

Empty turns per affected dialogue:
1    306
2    151
3     45
4     15
5      8
6      1
Name: count, dtype: int64


In [17]:
empty_user = empty_df[
    empty_df["speaker"] == "user"
].iloc[0]

print(empty_user)

source_data = sources[empty_user["source"]]

raw_turn = source_data[
    empty_user["dialogue_id"]
]["log"][empty_user["turn_id"]]

print("\nRaw turn:")
print(raw_turn)

dialogue = emowoz_df[
    (emowoz_df["split"] == empty_user["split"]) &
    (emowoz_df["source"] == empty_user["source"]) &
    (emowoz_df["dialogue_id"] == empty_user["dialogue_id"])
].sort_values("turn_id")

turn_id = empty_user["turn_id"]

dialogue[
    dialogue["turn_id"].between(turn_id - 3, turn_id + 3)
][
    ["turn_id", "speaker", "text", "emotion"]
]

split                  train
source              dialmage
dialogue_id    DMAGE812.json
turn_id                   18
speaker                 user
text                        
emotion_id                 2
emotion         dissatisfied
Name: 117740, dtype: object

Raw turn:
{'text': ' ', 'emotion': [{'annotator': 'c352279d', 'annotation': 2}, {'annotator': '5255f566', 'annotation': 2}, {'annotator': '64cfa354', 'annotation': 2}, {'emotion': 2, 'sentiment': 1}], 'dialog_act': {}, 'span_info': []}


,turn_id,speaker,text,emotion
117737,15,system,The phone is 01223358966 .,NaN
117738,16,user,ok,dissatisfied
117739,17,system,,NaN
117740,18,user,,dissatisfied
117741,19,system,The phone is 01223358966 .,NaN
117742,20,user,no,dissatisfied
117743,21,system,Thank you for using our services .,NaN


In [18]:
for split_name in ["train", "dev", "test"]:

    split_df = emowoz_df[
        emowoz_df["split"] == split_name
    ]

    n_dialogues = (
        split_df[
            ["source", "dialogue_id"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    n_turns = len(split_df)
    n_user = (split_df["speaker"] == "user").sum()
    n_system = (split_df["speaker"] == "system").sum()

    print(
        f"{split_name:5} | "
        f"{n_dialogues:,} dialogues | "
        f"{n_turns:,} turns | "
        f"{n_user:,} user | "
        f"{n_system:,} system"
    )

train | 9,233 dialogues | 132,948 turns | 66,474 user | 66,474 system
dev   | 1,100 dialogues | 17,018 turns | 8,509 user | 8,509 system
test  | 1,100 dialogues | 17,268 turns | 8,634 user | 8,634 system


In [19]:
print(f"Total turns: {len(emowoz_df):,}")

print(
    "User turns:",
    (emowoz_df["speaker"] == "user").sum()
)

print(
    "System turns:",
    (emowoz_df["speaker"] == "system").sum()
)

Total turns: 167,234
User turns: 83617
System turns: 83617


In [20]:
for split_name in ["train", "dev", "test"]:

    split_df = emowoz_df[
        emowoz_df["split"] == split_name
    ]

    n_dialogues = (
        split_df[["source", "dialogue_id"]]
        .drop_duplicates()
        .shape[0]
    )

    n_turns = len(split_df)
    n_user = (split_df["speaker"] == "user").sum()
    n_system = (split_df["speaker"] == "system").sum()

    print(
        f"{split_name:5} | "
        f"{n_dialogues:,} dialogues | "
        f"{n_turns:,} turns | "
        f"{n_user:,} user | "
        f"{n_system:,} system"
    )

print(f"\nTotal turns: {len(emowoz_df):,}")
print(
    "User turns:",
    (emowoz_df["speaker"] == "user").sum()
)
print(
    "System turns:",
    (emowoz_df["speaker"] == "system").sum()
)

train | 9,233 dialogues | 132,948 turns | 66,474 user | 66,474 system
dev   | 1,100 dialogues | 17,018 turns | 8,509 user | 8,509 system
test  | 1,100 dialogues | 17,268 turns | 8,634 user | 8,634 system

Total turns: 167,234
User turns: 83617
System turns: 83617


In [21]:
user_df = emowoz_df[
    emowoz_df["speaker"] == "user"
].copy()

emotion_order = [
    "neutral",
    "fearful",
    "dissatisfied",
    "apologetic",
    "abusive",
    "excited",
    "satisfied",
]

for split_name in ["train", "dev", "test"]:

    split_users = user_df[
        user_df["split"] == split_name
    ]

    counts = split_users["emotion"].value_counts()

    print(f"\n{split_name.upper()}")

    for emotion in emotion_order:
        count = counts.get(emotion, 0)

        print(
            f"{emotion:13} "
            f"{count:>6,} "
            f"({count / len(split_users) * 100:5.2f}%)"
        )


TRAIN
neutral       46,658 (70.19%)
fearful          359 ( 0.54%)
dissatisfied   4,046 ( 6.09%)
apologetic       683 ( 1.03%)
abusive           69 ( 0.10%)
excited          766 ( 1.15%)
satisfied     13,893 (20.90%)

DEV
neutral        5,984 (70.33%)
fearful           19 ( 0.22%)
dissatisfied     467 ( 5.49%)
apologetic        84 ( 0.99%)
abusive           19 ( 0.22%)
excited          114 ( 1.34%)
satisfied      1,822 (21.41%)

TEST
neutral        6,014 (69.65%)
fearful           18 ( 0.21%)
dissatisfied     604 ( 7.00%)
apologetic        73 ( 0.85%)
abusive           17 ( 0.20%)
excited           91 ( 1.05%)
satisfied      1,817 (21.04%)


In [22]:
counts = user_df["emotion"].value_counts()

print("\nOVERALL")

for emotion in emotion_order:
    count = counts.get(emotion, 0)

    print(
        f"{emotion:13} "
        f"{count:>6,} "
        f"({count / len(user_df) * 100:5.2f}%)"
    )


OVERALL
neutral       58,656 (70.15%)
fearful          396 ( 0.47%)
dissatisfied   5,117 ( 6.12%)
apologetic       840 ( 1.00%)
abusive          105 ( 0.13%)
excited          971 ( 1.16%)
satisfied     17,532 (20.97%)


In [23]:
forecast_rows = []

for key, dialogue in emowoz_df.groupby(
    ["split", "source", "dialogue_id"]
):
    split_name, source_name, dialogue_id = key

    dialogue = dialogue.sort_values("turn_id")

    user_turns = dialogue[
        dialogue["speaker"] == "user"
    ]

    rows = list(user_turns.itertuples())

    for current, nxt in zip(rows, rows[1:]):

        # In normal alternating structure:
        # user t -> system t+1 -> user t+2
        if nxt.turn_id != current.turn_id + 2:
            continue

        system_turn = dialogue[
            dialogue["turn_id"] == current.turn_id + 1
        ]

        if len(system_turn) != 1:
            continue

        system = system_turn.iloc[0]

        forecast_rows.append({
            "split": split_name,
            "source": source_name,
            "dialogue_id": dialogue_id,

            "current_turn_id": current.turn_id,
            "current_text": current.text,
            "current_emotion": current.emotion,

            "system_text": system["text"],

            "next_turn_id": nxt.turn_id,
            "next_text": nxt.text,
            "next_emotion": nxt.emotion,

            "system_text_empty":
                not str(system["text"]).strip(),

            "current_text_empty":
                not str(current.text).strip(),

            "next_text_empty":
                not str(nxt.text).strip(),
        })

forecast_df = pd.DataFrame(forecast_rows)

print(f"User → system → user sequences: {len(forecast_df):,}")

User → system → user sequences: 72,184


In [24]:
print(
    "Empty intervening system:",
    forecast_df["system_text_empty"].sum()
)

print(
    "Empty current user:",
    forecast_df["current_text_empty"].sum()
)

print(
    "Empty next user:",
    forecast_df["next_text_empty"].sum()
)

valid_forecast_df = forecast_df[
    ~forecast_df["system_text_empty"]
    & ~forecast_df["current_text_empty"]
    & ~forecast_df["next_text_empty"]
].copy()

print(
    f"\nFully text-valid forecasts: "
    f"{len(valid_forecast_df):,}"
)

Empty intervening system: 776
Empty current user: 1
Empty next user: 1

Fully text-valid forecasts: 71,407


In [25]:
same = (
    forecast_df["current_emotion"]
    == forecast_df["next_emotion"]
)

print(f"Forecasting pairs: {len(forecast_df):,}")

print(
    f"Same-state: {same.sum():,} "
    f"({same.mean() * 100:.2f}%)"
)

print(
    f"Change: {(~same).sum():,} "
    f"({(~same).mean() * 100:.2f}%)"
)

Forecasting pairs: 72,184
Same-state: 46,071 (63.82%)
Change: 26,113 (36.18%)


In [26]:
target_counts = forecast_df[
    "next_emotion"
].value_counts()

print(target_counts)

majority_class = target_counts.idxmax()
majority_accuracy = (
    target_counts.max() / len(forecast_df)
)

print(
    f"\nAlways {majority_class}: "
    f"{majority_accuracy * 100:.2f}%"
)

print(
    f"Predict previous user emotion again: "
    f"{same.mean() * 100:.2f}%"
)

next_emotion
neutral         47803
satisfied       17527
dissatisfied     5117
apologetic        838
excited           608
fearful           187
abusive           104
Name: count, dtype: int64

Always neutral: 66.22%
Predict previous user emotion again: 63.82%


In [27]:
dialogue_stats = []

for key, dialogue in user_df.groupby(
    ["split", "source", "dialogue_id"]
):

    dialogue = dialogue.sort_values("turn_id")
    emotions = dialogue["emotion"].tolist()

    changes = sum(
        a != b
        for a, b in zip(emotions, emotions[1:])
    )

    dialogue_stats.append({
        "split": key[0],
        "source": key[1],
        "dialogue_id": key[2],
        "n_user_turns": len(emotions),
        "n_unique_emotions": len(set(emotions)),
        "n_changes": changes,
        "all_neutral": all(
            e == "neutral" for e in emotions
        ),
    })

dialogue_stats = pd.DataFrame(dialogue_stats)

print(
    f"All-neutral dialogues: "
    f"{dialogue_stats['all_neutral'].sum():,} "
    f"({dialogue_stats['all_neutral'].mean()*100:.2f}%)"
)

print(
    f"Mean changes/dialogue: "
    f"{dialogue_stats['n_changes'].mean():.2f}"
)

print(
    f"Median changes/dialogue: "
    f"{dialogue_stats['n_changes'].median():.0f}"
)

print(
    f"Zero-change dialogues: "
    f"{(dialogue_stats['n_changes'] == 0).sum():,} "
    f"{(dialogue_stats['n_changes'] == 0).mean()*100:.2f}%"
)

print(
    f"At least one change: "
    f"{(dialogue_stats['n_changes'] > 0).sum():,} "
    f"{(dialogue_stats['n_changes'] > 0).mean()*100:.2f}%"
)

All-neutral dialogues: 550 (4.81%)
Mean changes/dialogue: 2.28
Median changes/dialogue: 2
Zero-change dialogues: 551 4.82%
At least one change: 10,882 95.18%


In [28]:
emotion_order = [
    "neutral",
    "fearful",
    "dissatisfied",
    "apologetic",
    "abusive",
    "excited",
    "satisfied",
]

transition_matrix = pd.crosstab(
    forecast_df["current_emotion"],
    forecast_df["next_emotion"]
).reindex(
    index=emotion_order,
    columns=emotion_order,
    fill_value=0
)

transition_matrix

next_emotion,neutral,fearful,dissatisfied,apologetic,abusive,excited,satisfied
current_emotion,,,,,,,
neutral,40142,139,2734,665,40,479,12953
fearful,246,22,14,2,0,3,105
dissatisfied,1977,10,2204,15,39,11,374
apologetic,505,1,10,12,0,6,259
abusive,36,0,30,1,17,0,6
excited,671,2,20,8,0,47,203
satisfied,4226,13,105,135,8,62,3627


In [29]:
transition_probs = (
    transition_matrix
    .div(transition_matrix.sum(axis=1), axis=0)
)

transition_probs.round(3)

next_emotion,neutral,fearful,dissatisfied,apologetic,abusive,excited,satisfied
current_emotion,,,,,,,
neutral,0.702,0.002,0.048,0.012,0.001,0.008,0.227
fearful,0.628,0.056,0.036,0.005,0.000,0.008,0.268
dissatisfied,0.427,0.002,0.476,0.003,0.008,0.002,0.081
apologetic,0.637,0.001,0.013,0.015,0.000,0.008,0.327
abusive,0.400,0.000,0.333,0.011,0.189,0.000,0.067
excited,0.706,0.002,0.021,0.008,0.000,0.049,0.213
satisfied,0.517,0.002,0.013,0.017,0.001,0.008,0.444


In [30]:
change_matrix = transition_matrix.copy()

for emotion in emotion_order:
    change_matrix.loc[emotion, emotion] = 0

total_changes = change_matrix.to_numpy().sum()

changes = []

for from_emotion in emotion_order:
    for to_emotion in emotion_order:

        count = change_matrix.loc[
            from_emotion,
            to_emotion
        ]

        if count > 0:
            changes.append({
                "from": from_emotion,
                "to": to_emotion,
                "count": count,
                "percentage": count / total_changes * 100,
            })

changes_df = (
    pd.DataFrame(changes)
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(f"Total emotion changes: {total_changes:,}")

changes_df.head(20)

Total emotion changes: 26,113


,from,to,count,percentage
0,neutral,satisfied,12953,49.603646
1,satisfied,neutral,4226,16.183510
2,neutral,dissatisfied,2734,10.469881
3,dissatisfied,neutral,1977,7.570942
4,excited,neutral,671,2.569601
5,neutral,apologetic,665,2.546624
6,apologetic,neutral,505,1.933903
7,neutral,excited,479,1.834335
8,dissatisfied,satisfied,374,1.432237
9,apologetic,satisfied,259,0.991843


In [31]:
from collections import Counter

diversity = Counter(
    dialogue_stats["n_unique_emotions"]
)

for n_unique, count in sorted(diversity.items()):
    print(
        f"{n_unique} unique emotion(s): "
        f"{count:,} dialogues "
        f"({count / len(dialogue_stats) * 100:.2f}%)"
    )

1 unique emotion(s): 551 dialogues (4.82%)
2 unique emotion(s): 8,210 dialogues (71.81%)
3 unique emotion(s): 2,379 dialogues (20.81%)
4 unique emotion(s): 285 dialogues (2.49%)
5 unique emotion(s): 8 dialogues (0.07%)
